In [ ]:
import os

# Path to your Drive (My Drive is the root folder)
folder_path = "/content/drive/MyDrive/VisionExtract"

# Create folder if it doesn't exist
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print("✅ Folder created:", folder_path)
else:
    print("⚡ Folder already exists:", folder_path)
import os

train_folder = "/content/drive/MyDrive/VisionExtract/train2017_subset"
val_folder = "/content/drive/MyDrive/VisionExtract/val2017"

train_images = [f for f in os.listdir(train_folder) if f.endswith(('.jpg', '.jpeg', '.png'))]
val_images = [f for f in os.listdir(val_folder) if f.endswith(('.jpg', '.jpeg', '.png'))]

print("Training images:", len(train_images))
print("Validation images:", len(val_images))

# =========================
# Step 0: Install pycocotools correctly
# =========================
!pip uninstall -y pycocotools
!pip install pycocotools --no-binary pycocotools

# =========================
# Step 1: Imports
# =========================
import os
import random
import matplotlib.pyplot as plt
import cv2
from pycocotools.coco import COCO

# ======================
# Step 2: Paths
# ======================
ANN_FILE = "/content/drive/MyDrive/VisionExtract/annotations/instances_val2017.json"
IMG_DIR = "/content/drive/MyDrive/VisionExtract/val2017"

# Load COCO dataset
coco = COCO(ANN_FILE)

# ======================
# Step 3: Pick a random image
# ======================
img_id = random.choice(coco.getImgIds())
img_info = coco.loadImgs(img_id)[0]

# Load image
img_path = os.path.join(IMG_DIR, img_info['file_name'])
image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

# Load annotations for this image
ann_ids = coco.getAnnIds(imgIds=img_id)
anns = coco.loadAnns(ann_ids)

# ======================
# Step 4: Plot Left (Original) + Right (With Outlines)
# ======================
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Left → Original Image
axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

# Right → Image with COCO segmentation outlines
plt.sca(axes[1])
plt.imshow(image)
coco.showAnns(anns)  # overlays segmentation outlines
plt.title("With Segmentation Outlines")
plt.axis("off")

plt.tight_layout()
plt.show()




In [ ]:
# =========================
# Step 0: Imports
# =========================
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils
import albumentations as A  # For image augmentations

# =========================
# Step 1: Paths
# =========================
VAL_FOLDER = "/content/drive/MyDrive/VisionExtract/val2017"
ANNOT_FILE = "/content/drive/MyDrive/VisionExtract/annotations/instances_val2017.json"

# Load COCO annotations
coco = COCO(ANNOT_FILE)

# =========================
# Step 2: Preprocessing Parameters
# =========================
IMG_HEIGHT = 256
IMG_WIDTH = 256

# Augmentation pipeline (optional)
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
], additional_targets={'mask': 'mask'})  # Ensures mask transforms same as image

# =========================
# Step 3: Preprocessing Functions
# =========================

def resize_image_and_mask(image, mask):
    """
    Resize image and mask to target size.
    Mask uses INTER_NEAREST to preserve binary values.
    """
    image_resized = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT))
    mask_resized = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_NEAREST)
    return image_resized, mask_resized

def normalize_image(image):
    """
    Normalize image pixels to [0,1] range.
    """
    return image / 255.0

def convert_to_binary_mask(mask):
    """
    Convert multi-class mask into binary mask (0 = background, 1 = subject)
    """
    return np.where(mask > 0, 1, 0).astype(np.uint8)

def preprocess_image_and_mask(image, mask):
    """
    Complete preprocessing pipeline:
    1. Resize image & mask
    2. Convert mask to binary
    3. Apply augmentation
    4. Normalize image
    """
    # Resize
    image, mask = resize_image_and_mask(image, mask)

    # Convert to binary mask
    mask = convert_to_binary_mask(mask)

    # Apply optional augmentation
    augmented = transform(image=image, mask=mask)
    image_aug, mask_aug = augmented['image'], augmented['mask']

    # Normalize image
    image_aug = normalize_image(image_aug)

    return image_aug, mask_aug

# =========================
# Step 4: Function to create combined mask
# =========================
def create_combined_mask(img_id):
    """
    Create a single combined mask for all objects in the image.
    """
    img_info = coco.loadImgs(img_id)[0]
    mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)

    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    for ann in anns:
        if 'segmentation' in ann:
            rle = mask_utils.frPyObjects(ann['segmentation'], img_info['height'], img_info['width'])
            decoded_mask = mask_utils.decode(rle)

            # If mask is 3D, sum across channels
            if decoded_mask.ndim == 3:
                decoded_mask = np.sum(decoded_mask, axis=2)

            mask += decoded_mask

    # Clip values to binary 0-1
    mask = np.clip(mask, 0, 1)

    return mask

# =========================
# Step 5: Preprocess & visualize multiple validation images
# =========================
NUM_IMAGES = 15  # Number of random images to visualize
random_val_ids = random.sample(coco.getImgIds(), NUM_IMAGES)

for idx, img_id in enumerate(random_val_ids):
    img_info = coco.loadImgs(img_id)[0]
    img_path = os.path.join(VAL_FOLDER, img_info['file_name'])

    if not os.path.exists(img_path):
        continue

    # Load image
    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    # Create combined mask
    mask = create_combined_mask(img_id)

    # Preprocess image & mask
    img_processed, mask_processed = preprocess_image_and_mask(image, mask)

    # Visualize
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.imshow(img_processed)
    plt.title(f"Image")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(mask_processed, cmap='gray')
    plt.title(f"Binary Mask ")
    plt.axis("off")

    plt.show()
# =========================
# Visualize
# =========================
NUM_IMAGES = 5  # visualize only 5
random_val_ids = random.sample(coco.getImgIds(), NUM_IMAGES)

for idx, img_id in enumerate(random_val_ids):
    img_info = coco.loadImgs(img_id)[0]
    img_path = os.path.join(VAL_FOLDER, img_info['file_name'])

    if not os.path.exists(img_path):
        continue

    # Load image
    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    # Create combined mask
    mask = create_combined_mask(img_id)

    # Preprocess image & mask
    img_processed, mask_processed = preprocess_image_and_mask(image, mask)

    # Visualize
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.imshow(img_processed)
    plt.title("Preprocessed Image")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(mask_processed, cmap='gray')
    plt.title("Binary Mask")
    plt.axis("off")

    plt.show()



In [ ]:
# =========================
# Step 0: Imports
# =========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights
import cv2
import numpy as np
import os
import random
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils

# =========================
# Step 1: Dataset Definition
# =========================
class SegmentationDataset(Dataset):
    def __init__(self, img_ids, img_folder, coco, img_size=(256,256)):
        self.img_ids = img_ids
        self.img_folder = img_folder
        self.coco = coco
        self.img_size = img_size

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_folder, img_info['file_name'])

        # Read and preprocess image
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, self.img_size)
        image = image / 255.0  # normalize
        image = np.transpose(image, (2,0,1)).astype(np.float32)  # C,H,W

        # Create combined mask (binary)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        for ann in anns:
            if 'segmentation' in ann:
                rle = mask_utils.frPyObjects(ann['segmentation'], img_info['height'], img_info['width'])
                decoded = mask_utils.decode(rle)
                if decoded.ndim == 3:
                    decoded = np.sum(decoded, axis=2)
                mask += decoded
        mask = np.clip(mask, 0, 1)
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
        mask = np.expand_dims(mask, axis=0).astype(np.float32)  # 1,H,W

        return torch.tensor(image), torch.tensor(mask)

# =========================
# Step 2: Load dataset
# =========================
VAL_FOLDER = "/content/drive/MyDrive/VisionExtract/val2017"
ANNOT_FILE = "/content/drive/MyDrive/VisionExtract/annotations/instances_val2017.json"

coco = COCO(ANNOT_FILE)

# Use 50 random images for demo training
img_ids = random.sample(coco.getImgIds(), 50)
dataset = SegmentationDataset(img_ids, VAL_FOLDER, coco)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# =========================
# Step 3: Model
# =========================
weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
model = torchvision.models.segmentation.deeplabv3_resnet50(weights=weights)

# Modify classifier for binary mask
model.classifier[-1] = nn.Conv2d(256, 1, kernel_size=(1,1))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# =========================
# Step 4: Loss and Optimizer
# =========================
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# =========================
# Step 5: Training Loop (demo: 2 epochs)
# =========================
for epoch in range(2):
    model.train()
    running_loss = 0.0
    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(dataset)
    print(f"Epoch [{epoch+1}/2], Loss: {epoch_loss:.4f}")

# =========================
# Step 6: Visualize Predictions
# =========================
model.eval()
with torch.no_grad():
    sample_img, sample_mask = dataset[0]
    input_img = sample_img.unsqueeze(0).to(device)
    pred = model(input_img)['out']
    pred_mask = torch.sigmoid(pred).cpu().squeeze().numpy()
    pred_mask_bin = (pred_mask > 0.5).astype(np.uint8)

    plt.figure(figsize=(10,5))
    plt.subplot(1,3,1)
    plt.imshow(np.transpose(sample_img, (1,2,0)))
    plt.title("Input Image")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(sample_mask.squeeze(), cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(pred_mask_bin, cmap='gray')
    plt.title("Prediction")
    plt.axis("off")
    plt.show()



In [ ]:
import torch

# Save full model
full_model_path = "/content/drive/MyDrive/VisionExtract/deeplabv3_binary_model.pth"
torch.save(model, full_model_path)
print(f"Full model saved at: {full_model_path}")

# Save only model weights
weights_path = "/content/drive/MyDrive/VisionExtract/deeplabv3_binary_weights.pth"
torch.save(model.state_dict(), weights_path)
print(f"Model weights saved at: {weights_path}")


In [ ]:
# =========================
# Step 0: Imports
# =========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import cv2
import numpy as np
import os
import random
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils
import albumentations as A

# =========================
# Step 1: Paths and COCO
# =========================
VAL_FOLDER = "/content/drive/MyDrive/VisionExtract/val2017"
ANNOT_FILE = "/content/drive/MyDrive/VisionExtract/annotations/instances_val2017.json"
coco = COCO(ANNOT_FILE)

# =========================
# Step 2: Dataset
# =========================
IMG_HEIGHT, IMG_WIDTH = 256, 256

class SegmentationDataset(Dataset):
    def __init__(self, img_ids, img_folder, coco, augment=False):
        self.img_ids = img_ids
        self.img_folder = img_folder
        self.coco = coco
        self.augment = augment
        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.HueSaturationValue(p=0.5),
        ], additional_targets={'mask':'mask'})

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_folder, img_info['file_name'])
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        for ann in anns:
            if 'segmentation' in ann:
                rle = mask_utils.frPyObjects(ann['segmentation'], img_info['height'], img_info['width'])
                decoded = mask_utils.decode(rle)
                if decoded.ndim == 3: decoded = np.sum(decoded, axis=2)
                mask += decoded
        mask = np.clip(mask, 0, 1)
        image = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT))
        mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_NEAREST)
        if self.augment:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']
        image = np.transpose(image / 255.0, (2,0,1)).astype(np.float32)
        mask = np.expand_dims(mask.astype(np.float32), axis=0)
        return torch.tensor(image), torch.tensor(mask)

# =========================
# Step 3: Train/Validation Split (subset for speed)
# =========================
all_img_ids = coco.getImgIds()
random.shuffle(all_img_ids)
split = int(0.8 * len(all_img_ids))
train_ids, val_ids = all_img_ids[:split], all_img_ids[split:]

# Subset for fast testing
train_subset_ids = train_ids[:100]  # increased training images
val_subset_ids = val_ids[:40]       # increased validation images

train_dataset = SegmentationDataset(train_subset_ids, VAL_FOLDER, coco, augment=True)
val_dataset = SegmentationDataset(val_subset_ids, VAL_FOLDER, coco, augment=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# =========================
# Step 4: Model
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = torchvision.models.segmentation.DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
model = torchvision.models.segmentation.deeplabv3_resnet50(weights=weights)
model.classifier[-1] = nn.Conv2d(256, 1, kernel_size=(1,1))  # binary output
model = model.to(device)

# =========================
# Step 5: Loss & Optimizer
# =========================
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# =========================
# Step 6: Metrics Function
# =========================
def compute_metrics(y_true, y_pred):
    y_pred_bin = (y_pred > 0.5).astype(np.uint8)
    # Pixel Accuracy
    pixel_acc = np.sum(y_true == y_pred_bin) / (y_true.shape[0] * y_true.shape[1])
    # IoU
    intersection = np.logical_and(y_true, y_pred_bin).sum()
    union = np.logical_or(y_true, y_pred_bin).sum()
    iou = intersection / (union + 1e-7)
    # Dice
    dice = (2 * intersection) / (y_true.sum() + y_pred_bin.sum() + 1e-7)
    return pixel_acc, iou, dice

# =========================
# Step 7: Training Loop
# =========================
num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    all_pixel_acc, all_ious, all_dices = [], [], []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)['out']
            loss = criterion(outputs, masks)
            val_loss += loss.item() * images.size(0)

            preds = torch.sigmoid(outputs).cpu().numpy()
            masks_np = masks.cpu().numpy()
            for i in range(preds.shape[0]):
                pixel_acc, iou, dice = compute_metrics(masks_np[i,0], preds[i,0])
                all_pixel_acc.append(pixel_acc)
                all_ious.append(iou)
                all_dices.append(dice)
    val_loss /= len(val_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
          f"Pixel Acc: {np.mean(all_pixel_acc):.4f}, Mean IoU: {np.mean(all_ious):.4f}, Dice: {np.mean(all_dices):.4f}")

# =========================
# Step 8: Visualize Predictions
# =========================
model.eval()
sample_val_ids = random.sample(val_subset_ids, 5)
with torch.no_grad():
    for img_id in sample_val_ids:
        idx = val_subset_ids.index(img_id)
        img_tensor, mask_tensor = val_dataset[idx]
        input_img = img_tensor.unsqueeze(0).to(device)
        pred = model(input_img)['out']
        pred_mask = torch.sigmoid(pred).cpu().squeeze().numpy()
        # Post-processing to remove noise
        pred_mask_bin = (pred_mask > 0.5).astype(np.uint8)
        pred_mask_bin = cv2.morphologyEx(pred_mask_bin, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5)))
        pred_mask_bin = cv2.morphologyEx(pred_mask_bin, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5)))

        plt.figure(figsize=(12,4))
        plt.subplot(1,3,1)
        plt.imshow(np.transpose(img_tensor, (1,2,0)))
        plt.title("Image"); plt.axis("off")

        plt.subplot(1,3,2)
        plt.imshow(mask_tensor.squeeze(), cmap='gray')
        plt.title("Ground Truth"); plt.axis("off")

        plt.subplot(1,3,3)
        plt.imshow(pred_mask_bin, cmap='gray')
        plt.title("Predicted Image"); plt.axis("off")
        plt.show()

# Path to save
MODEL_PATH = "/content/drive/MyDrive/VisionExtract/deeplabv3_subject_model.pth"

# Save only the model weights (recommended)
torch.save(model.state_dict(), MODEL_PATH)
print("Model saved successfully!")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2

# =========================
# Assume model is already loaded as `loaded_model`
# =========================

loaded_model.eval()

# Sample some validation images
sample_val_ids = random.sample(val_subset_ids, 15)

with torch.no_grad():
    for img_id in sample_val_ids:
        idx = val_subset_ids.index(img_id)
        img_tensor, mask_tensor = val_dataset[idx]
        input_img = img_tensor.unsqueeze(0).to(device)

        # Predict mask
        pred = loaded_model(input_img)['out']
        pred_mask = torch.sigmoid(pred).cpu().squeeze().numpy()

        # Binarize and post-process mask
        pred_mask_bin = (pred_mask > 0.5).astype(np.uint8)
        pred_mask_bin = cv2.morphologyEx(pred_mask_bin, cv2.MORPH_OPEN,
                                        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5)))
        pred_mask_bin = cv2.morphologyEx(pred_mask_bin, cv2.MORPH_CLOSE,
                                        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5)))

        # Convert tensors to original RGB image
        original_img = np.transpose(img_tensor.numpy(), (1,2,0))
        original_img = (original_img * 255).astype(np.uint8)

        # Create subject-isolated image
        subject_img = original_img * pred_mask_bin[:, :, np.newaxis]

        # Visualization
        plt.figure(figsize=(15,5))

        plt.subplot(1,3,1)
        plt.imshow(original_img)
        plt.title("Original Image")
        plt.axis("off")

        plt.subplot(1,3,2)
        plt.imshow(mask_tensor.squeeze(), cmap='gray')
        plt.title("Ground Truth Mask")
        plt.axis("off")

        plt.subplot(1,3,3)
        plt.imshow(subject_img)
        plt.title("Subject Isolated Image")
        plt.axis("off")

        plt.show()


In [ ]:

import torch
import torch.nn as nn
import torchvision
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
from PIL import Image
from pathlib import Path

# =========================
# Device & Load Model
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = torchvision.models.segmentation.DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
model = torchvision.models.segmentation.deeplabv3_resnet50(weights=weights)
model.classifier[-1] = nn.Conv2d(256, 1, kernel_size=(1,1))

MODEL_PATH = "/content/drive/MyDrive/VisionExtract/deeplabv3_subject_model.pth"
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

# =========================
# Post-processing function
# =========================
def postprocess_mask(mask):
    mask_bin = (mask > 0.5).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask_bin = (labels == largest_label).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_CLOSE, kernel)
    mask_bin = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN, kernel)
    mask_bin = cv2.GaussianBlur(mask_bin.astype(np.float32), (5,5), 0)
    mask_bin = (mask_bin > 0.5).astype(np.uint8)
    return mask_bin

# =========================
# RGB overlay (optional visualization)
# =========================
def apply_rgb_mask(image, mask, color=(0,255,0), alpha=0.5):
    rgb_mask = np.zeros_like(image, dtype=np.uint8)
    rgb_mask[mask==1] = color
    overlay = cv2.addWeighted(image, 1.0, rgb_mask, alpha, 0)
    return overlay

# =========================
# Inference function
# =========================
def inference(image_path, save_dir="output", overlay=True):
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    # Load image
    original_img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(original_img, (256,256))
    img_norm = np.transpose(img_resized / 255.0, (2,0,1)).astype(np.float32)
    img_tensor = torch.tensor(img_norm).unsqueeze(0).to(device)

    # Predict mask
    with torch.no_grad():
        pred = model(img_tensor)['out']
        pred_mask = torch.sigmoid(pred).cpu().squeeze().numpy()
    pred_mask_resized = cv2.resize(pred_mask, (original_img.shape[1], original_img.shape[0]), interpolation=cv2.INTER_NEAREST)
    pred_mask_clean = postprocess_mask(pred_mask_resized)

    # Subject-isolated image
    isolated_subject = original_img * np.expand_dims(pred_mask_clean, axis=2)

    # Save isolated image
    base_name = os.path.basename(image_path)
    save_path = os.path.join(save_dir, f"isolated_{base_name}")
    Image.fromarray(isolated_subject).save(save_path)

    # Optional RGB overlay
    if overlay:
        overlay_img = apply_rgb_mask(original_img, pred_mask_clean, color=(0,255,0), alpha=0.5)
        overlay_path = os.path.join(save_dir, f"overlay_{base_name}")
        Image.fromarray(overlay_img).save(overlay_path)

    return original_img, pred_mask_clean, isolated_subject

# =========================
# Step 4: Process all input images and show outputs
# =========================
input_folder = "/content/drive/MyDrive/VisionExtract/test_images/Input_Images"
output_folder = "/content/drive/MyDrive/VisionExtract/test_images/Output_Images"
Path(output_folder).mkdir(parents=True, exist_ok=True)

all_images = [os.path.join(input_folder, f) for f in os.listdir(input_folder)
              if f.lower().endswith((".png", ".jpg", ".jpeg"))]

for img_path in all_images:
    original, mask, isolated = inference(img_path, save_dir=output_folder)

    # =========================
    # Display images
    # =========================
    plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.imshow(original)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(mask, cmap='gray')
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(isolated)
    plt.title("Subject Isolated Output")
    plt.axis("off")

    plt.show()
